# 장르별 가격 포지셔닝 분석

**분석 질문**
- 장르별로 어떤 가격대에 게임이 집중되어 있는가?
- 같은 장르에서 가격대별 평균 긍정률은 어떻게 다른가?
- 출시 전 가격 설정 시 참고할 수 있는 장르별 최적 가격대는?

In [ ]:
import ast
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

print('라이브러리 로드 완료')

In [2]:
DATA_PATH = Path('../../../data/preprocessed/steam_indie_games.csv')
df_games = pd.read_csv(DATA_PATH)
print(f'데이터 로드 완료: {df_games.shape[0]:,}개 게임')

데이터 로드 완료: 9,169개 게임


In [3]:
def parse_genres(value: str) -> list[str]:
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in str(value).split(',') if item.strip()]


df_clean = df_games.copy()
for col in ['positive', 'negative', 'total_reviews', 'price']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean = df_clean.dropna(subset=['positive', 'negative', 'total_reviews', 'price'])
df_clean = df_clean[df_clean['total_reviews'] > 0].copy()
df_clean['positive_rate'] = df_clean['positive'] / df_clean['total_reviews'] * 100
df_clean['genre_list'] = df_clean['genres'].apply(parse_genres)
df_clean = df_clean[df_clean['genre_list'].map(len) > 0].copy()

# 가격 구간 설정
price_bins = [0, 5, 10, 15, 20, 30, 50, float('inf')]
price_labels = ['~5$', '5~10$', '10~15$', '15~20$', '20~30$', '30~50$', '50$~']
df_clean['price_range'] = pd.cut(df_clean['price'], bins=price_bins, labels=price_labels, right=True)

print(f'분석 가능 게임 수: {len(df_clean):,}개')
print('가격 구간별 게임 수:')
display(df_clean['price_range'].value_counts().sort_index().to_frame('게임 수'))

분석 가능 게임 수: 9,169개
가격 구간별 게임 수:


,게임 수
price_range,
~5$,3910
5~10$,2554
10~15$,1275
15~20$,826
20~30$,341
30~50$,68
50$~,23


In [4]:
# 장르 × 가격 분석을 위해 explode
df_genre = df_clean.explode('genre_list').rename(columns={'genre_list': 'genre'})
df_genre['genre'] = df_genre['genre'].astype(str).str.strip()
df_genre = df_genre[(df_genre['genre'] != '') & (df_genre['genre'] != 'Indie')].copy()

MIN_GAMES = 30
valid_genres = (
    df_genre.groupby('genre')['appid'].nunique()
    .pipe(lambda s: s[s >= MIN_GAMES].index.tolist())
)
df_genre = df_genre[df_genre['genre'].isin(valid_genres)].copy()

print(f'분석 대상 장르: {sorted(valid_genres)}')

분석 대상 장르: ['Action', 'Adventure', 'Casual', 'RPG', 'Racing', 'Simulation', 'Sports', 'Strategy']


## 장르별 가격 분포

In [ ]:
fig = px.box(
    df_genre,
    x='genre',
    y='price',
    color='genre',
    title='장르별 가격 분포',
    labels={'genre': '장르', 'price': '가격 ($)'},
    range_y=[0, 60],
)
fig.update_layout(showlegend=False, height=450)
fig.show()

## 장르 × 가격대별 평균 긍정률 히트맵

In [ ]:
MIN_GAMES_PER_CELL = 5

heatmap_data = (
    df_genre
    .groupby(['genre', 'price_range'], observed=True)
    .agg(avg_positive_rate=('positive_rate', 'mean'), count=('appid', 'nunique'))
    .reset_index()
)
heatmap_data.loc[heatmap_data['count'] < MIN_GAMES_PER_CELL, 'avg_positive_rate'] = float('nan')

pivot = heatmap_data.pivot(index='genre', columns='price_range', values='avg_positive_rate').round(1)

fig = px.imshow(
    pivot,
    text_auto='.1f',
    color_continuous_scale='RdYlGn',
    zmin=70,
    zmax=95,
    title=f'장르 × 가격대별 평균 긍정률 (게임 수 {MIN_GAMES_PER_CELL}개 미만 셀 제외)',
    labels={'x': '가격대', 'y': '장르', 'color': '평균 긍정률 (%)'},
    aspect='auto',
)
fig.update_layout(height=400)
fig.show()

## 장르별 가격대 게임 수 분포

In [ ]:
count_data = (
    df_genre
    .groupby(['genre', 'price_range'], observed=True)['appid']
    .nunique()
    .reset_index(name='game_count')
)

fig = px.bar(
    count_data,
    x='genre',
    y='game_count',
    color='price_range',
    barmode='group',
    title='장르별 가격대 게임 수 분포',
    labels={'genre': '장르', 'game_count': '게임 수', 'price_range': '가격대'},
)
fig.update_layout(height=450)
fig.show()

## 요약 테이블

In [8]:
summary = (
    df_genre
    .groupby('genre')
    .agg(
        game_count=('appid', 'nunique'),
        median_price=('price', 'median'),
        avg_price=('price', 'mean'),
        avg_positive_rate=('positive_rate', 'mean'),
    )
    .round(2)
    .sort_values('avg_positive_rate', ascending=False)
    .reset_index()
)

print('장르별 가격 및 긍정률 요약')
display(summary)

장르별 가격 및 긍정률 요약


,genre,game_count,median_price,avg_price,avg_positive_rate
0,Casual,4051,4.99,7.95,85.19
1,Action,3995,6.99,9.48,83.69
2,Adventure,4732,6.99,9.52,83.59
3,Strategy,1980,7.99,10.91,83.02
4,Racing,284,5.99,13.09,82.97
5,Sports,329,6.99,14.84,82.84
6,RPG,2155,8.24,10.89,82.42
7,Simulation,2429,6.99,10.15,79.78
